### Google Colab Connection

In [1]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

REP0_PATH = "/content/drive/MyDrive/Brick_Detection/"

REPO_DIR = os.path.join(REP0_PATH, "repo")

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    !git clone https://github.com/Crightub/lego-brick-detection.git {REPO_DIR}
else:
    print("Repo already exists, pulling latest changes...")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

# 3. Install dependencies
print("Installing requirements...")
!pip install -r requirements.txt -q
print("Done!")


Mounted at /content/drive
Repo already exists, pulling latest changes...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 3 (delta 1), reused 3 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 492 bytes | 7.00 KiB/s, done.
From https://github.com/Crightub/lego-brick-detection
   64340a0..7b6f5d5  main       -> origin/main
Updating 64340a0..7b6f5d5
Fast-forward
 requirements.txt | 128 ++++++++-----------------------------------------------
 1 file changed, 17 insertions(+), 111 deletions(-)
Working directory: /content/drive/MyDrive/Brick_Detection/repo
Installing requirements...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 62.6 MB/

### Local Execution

In [1]:
import shutil
import os
from sklearn.model_selection import train_test_split
import csv
import cv2
from tqdm import tqdm
import xml.etree.ElementTree as ET

REP0_PATH = os.getcwd().join('..')

### Dataset Preparation


In [2]:
image_path = os.path.join(REP0_PATH, "B200_Lego_Dataset/images")
annotation_path = os.path.join(REP0_PATH, "B200_Lego_Dataset/annotations")
label_path = os.path.join(REP0_PATH, "data/labels")

### Scale Down of Pictures

The original picture have a size of 2000x2000 which results in a large data size.
We scale down the pictures to a size of 640x640 for more efficient training of the stage 1 object detection.

In [3]:
output_scaled_base_path = os.path.join(REP0_PATH, "data/scaled_data")
output_scaled_image_path = os.path.join(output_scaled_base_path, "images")
output_scaled_annotation_path = os.path.join(output_scaled_base_path, "annotations")

os.makedirs(output_scaled_base_path, exist_ok=True)
os.makedirs(output_scaled_image_path, exist_ok=True)
os.makedirs(output_scaled_annotation_path, exist_ok=True)

TARGET_SIZE = 640


def downscale_dataset():
    image_files = [f for f in os.listdir(image_path) if f.lower().endswith(('.jpg', '.png'))]

    for img_name in tqdm(image_files):
        img = cv2.imread(os.path.join(image_path, img_name))
        h_orig, w_orig = img.shape[:2]

        scale_x = TARGET_SIZE / w_orig
        scale_y = TARGET_SIZE / h_orig

        img_resized = cv2.resize(img, (TARGET_SIZE, TARGET_SIZE))

        xml_name = img_name.rsplit('.', 1)[0] + '.xml'
        tree = ET.parse(os.path.join(annotation_path, xml_name))
        root = tree.getroot()

        for obj in root.findall('object'):
            bbox = obj.find('bndbox')
            xmin = float(bbox.find('xmin').text) * scale_x
            ymin = float(bbox.find('ymin').text) * scale_y
            xmax = float(bbox.find('xmax').text) * scale_x
            ymax = float(bbox.find('ymax').text) * scale_y

            # CRITICAL: Ensure no zero-area boxes
            if (xmax - xmin) < 1.0: xmax += 1.0
            if (ymax - ymin) < 1.0: ymax += 1.0

            bbox.find('xmin').text = str(round(xmin, 2))
            bbox.find('ymin').text = str(round(ymin, 2))
            bbox.find('xmax').text = str(round(xmax, 2))
            bbox.find('ymax').text = str(round(ymax, 2))

        cv2.imwrite(os.path.join(output_scaled_image_path, img_name), img_resized)
        tree.write(os.path.join(output_scaled_annotation_path, xml_name))


downscale_dataset()

FileNotFoundError: [Errno 2] No such file or directory: './Users/pfisterer/PycharmProjects/BrickDetection/DatasetPrep./B200_Lego_Dataset/images'

In [ ]:
if not os.path.exists(output_scaled_base_path):
    downscale_dataset()
else:
    print('Preprocessed data found.')

### Train, Val, Test Split

Generate a train, val, test split with percentages (0.7, 0.15, 0.15) resulting in samples size of (1400, 300, 300).
To reuse the same split later and compare different models, store the samples in the subdirectory /train, /val, /test.

In [4]:
def move_files(files, source, destination):
    os.makedirs(destination, exist_ok=True)
    for f in files:
        shutil.copy(os.path.join(source, f), os.path.join(destination, f))


def generate_train_val_test_split(image_dir, annotation_dir, output_dir):
    images = sorted(os.listdir(image_dir))
    annotations = sorted(os.listdir(annotation_dir))

    out_train_img = os.path.join(output_dir, 'train/images')
    out_train_annotation = os.path.join(output_dir, 'train/annotations')
    out_val_img = os.path.join(output_dir, 'val/images')
    out_val_annotation = os.path.join(output_dir, 'val/annotations')
    out_test_img = os.path.join(output_dir, 'test/images')
    out_test_annotation = os.path.join(output_dir, 'test/annotations')

    images_train, images_temp, annotations_train, annotations_temp = train_test_split(images, annotations,
                                                                                      train_size=0.7,
                                                                                      test_size=0.3, random_state=42)
    images_val, images_test, annotations_val, annotations_test = train_test_split(images_temp, annotations_temp,
                                                                                  train_size=0.5, test_size=0.5,
                                                                                  random_state=42)

    move_files(images_train, image_dir, out_train_img)
    move_files(annotations_train, annotation_dir, out_train_annotation)

    move_files(images_val, image_dir, out_val_img)
    move_files(annotations_val, annotation_dir, out_val_annotation)

    move_files(images_test, image_dir, out_test_img)
    move_files(annotations_test, annotation_dir, out_test_annotation)

In [ ]:
# Generate Split for downscaled images
generate_train_val_test_split(output_scaled_image_path, output_scaled_annotation_path, output_scaled_base_path)

In [5]:
# Generate Split for original images
out = os.getcwd() + "/../data/"

output_image_path = os.getcwd() + '/../data/images'
output_annotation_path = os.getcwd() + '/../data/annotations'

generate_train_val_test_split(output_image_path, output_annotation_path, out)

### Create Label Mapping

Object detection models expect labels starting from 0,...,m with 0 representing the background.
Create a mapping from the lego brick id to a label from {1,...,m} and store it in a csv file.
The dataset while later use this to identify the number of labels and convert the brick id in the annotation to a label.

In [ ]:
labels_path = os.path.join(REP0_PATH, 'labels_map')


def generate_labels_map():
    labels_map = {}
    label_counter = 1
    for file in os.listdir(annotation_path):
        tree = ET.parse(os.path.join(annotation_path, file))
        root = tree.getroot()
        for obj in root.iter('object'):
            label = obj.find('name').text
            if label not in labels_map and label_counter < 20:
                labels_map[label] = label_counter
                label_counter += 1

    with open(labels_path, "w", newline='') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerows(labels_map.items())

In [ ]:
if not os.path.exists(labels_path):
    generate_labels_map()
else:
    print('Labels map data found.')

Yolo Conversion & Crop Extraction

In [7]:
from DatasetPrep.yolo_conversion import convert_annotations_to_yolo
from DatasetPrep.crop_extraction import extract_crops

In [1]:
convert_annotations_to_yolo('../data/train/annotations', 'data/train/yolo_labes')
convert_annotations_to_yolo('../data/val/annotations', 'data/val/yolo_labes')
convert_annotations_to_yolo('../data/test/annotations', 'data/test/yolo_labes')

Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400
Objects: 400

In [8]:
extract_crops(
    image_dir='../data/train/images',
    annotation_dir='../data/train/annotations',
    labels_path='../data/labels_map.csv',
    out_dir='../data/train/crops'
)
extract_crops(
    image_dir='../data/val/images',
    annotation_dir='../data/val/annotations',
    labels_path='../data/labels_map.csv',
    out_dir='../data/val/crops'
)

Extracted 560000 crops to ../data/train/crops
Extracted 120000 crops to ../data/val/crops
